# Cross-Axis Correlation Prep (ECFP4, MACCS, MAP4)

This notebook computes descriptor-bit statistical co-occurrence matrices on the **training set only**.

Outputs produced for each fingerprint type:
1. Point-biserial (Pearson-equivalent) correlation matrix between bits and Axis 1 descriptors.
2. Best-matching descriptor index per bit using argmax over absolute correlation.

Fingerprint targets in this notebook:
- ECFP4: 2048 bits
- MACCS: 166 bits (RDKit bit 0 excluded)
- MAP4: 1024 bits (binarized for bit-wise correlation)

No SMARTS recovery is used here; this is purely statistical co-occurrence.

In [1]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm

from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys

try:
    from map4 import MAP4
    MAP4_AVAILABLE = True
except Exception:
    MAP4_AVAILABLE = False

pd.set_option('display.max_columns', 200)

In [2]:
# -----------------------------
# Config
# -----------------------------
ROOT = Path('..')
DATA_DIR = ROOT / 'data' / 'processed'
RESULTS_DIR = ROOT / 'results'
OUT_DIR = RESULTS_DIR / 'cross_axis_correlation_prep'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DESCRIPTOR_TABLE_PATH = DATA_DIR / 'massspecgym_complete' / 'all_rdkit_descriptors.parquet'
AXIS1_R2_PATH = RESULTS_DIR / 'indicators' / 'probe_indicator_merged.csv'
TRAIN_HDF5_PATH = DATA_DIR / 'MassSpecGym_splits' / 'finetuning.hdf5'

ECFP4_BITS = 2048
MACCS_BITS = 166
MAP4_BITS = 1024
ECFP4_RADIUS = 2
MAP4_RADIUS = 2

print('Using paths:')
print(f'  Descriptor table: {DESCRIPTOR_TABLE_PATH}')
print(f'  Axis 1 R2 table:  {AXIS1_R2_PATH}')
print(f'  Training HDF5:    {TRAIN_HDF5_PATH}')
print(f'  Output dir:       {OUT_DIR}')
print(f'  MAP4 available:   {MAP4_AVAILABLE}')

Using paths:
  Descriptor table: ../data/processed/massspecgym_complete/all_rdkit_descriptors.parquet
  Axis 1 R2 table:  ../results/indicators/probe_indicator_merged.csv
  Training HDF5:    ../data/processed/MassSpecGym_splits/finetuning.hdf5
  Output dir:       ../results/cross_axis_correlation_prep
  MAP4 available:   True


In [3]:
# -----------------------------
# Load Axis 1 descriptor list and align training molecules
# -----------------------------
axis1 = pd.read_csv(AXIS1_R2_PATH)
if not {'descriptor', 'r2_linear'}.issubset(axis1.columns):
    raise ValueError('Axis 1 file must contain descriptor and r2_linear columns.')

axis1 = axis1[['descriptor', 'r2_linear']].dropna(subset=['descriptor']).copy()
axis1 = axis1.sort_values('descriptor').drop_duplicates(subset='descriptor', keep='first')
axis1_descriptor_order = axis1['descriptor'].tolist()

df_desc_all = pd.read_parquet(DESCRIPTOR_TABLE_PATH)
if 'smiles' not in df_desc_all.columns:
    raise ValueError('Descriptor table must contain a smiles column for alignment.')

descriptor_cols = [d for d in axis1_descriptor_order if d in df_desc_all.columns]
missing_desc = [d for d in axis1_descriptor_order if d not in df_desc_all.columns]
if missing_desc:
    print(f'Warning: {len(missing_desc)} Axis 1 descriptors missing from descriptor table.')

# Keep one descriptor row per SMILES to avoid duplicated molecules
df_desc = df_desc_all[['smiles'] + descriptor_cols].drop_duplicates(subset=['smiles'], keep='first').copy()

# Load training molecules from finetuning split
with h5py.File(TRAIN_HDF5_PATH, 'r') as f:
    smiles_raw = f['smiles'][:]
    fold_raw = f['fold'][:]

train_smiles = [s.decode('utf-8') if isinstance(s, bytes) else str(s) for s in smiles_raw]
train_folds = [x.decode('utf-8') if isinstance(x, bytes) else str(x) for x in fold_raw]

# Preserve first-seen order of training molecules
seen = set()
train_smiles_ordered = []
for smi, fold in zip(train_smiles, train_folds):
    if fold == 'train' and smi not in seen:
        train_smiles_ordered.append(smi)
        seen.add(smi)

df_train = pd.DataFrame({'smiles': train_smiles_ordered})

# Align descriptors to the training molecule order
df = df_train.merge(df_desc, on='smiles', how='inner')
df = df.dropna(subset=descriptor_cols).reset_index(drop=True)

X_desc = df[descriptor_cols].to_numpy(dtype=np.float32)
N = X_desc.shape[0]
D = X_desc.shape[1]

print(f'Training unique molecules in HDF5: {len(train_smiles_ordered)}')
print(f'Aligned molecules after merge/dropna: {N}')
print(f'Descriptor columns used: {D}')

Training unique molecules in HDF5: 18251
Aligned molecules after merge/dropna: 18242
Descriptor columns used: 201


In [4]:
# -----------------------------
# Build fingerprint bit matrices aligned to descriptor rows
# -----------------------------
smiles_list = df['smiles'].astype(str).tolist()

def ecfp4_bits_from_smiles(smiles_str, n_bits=ECFP4_BITS, radius=ECFP4_RADIUS):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return np.zeros(n_bits, dtype=np.uint8)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.uint8)
    arr[list(fp.GetOnBits())] = 1
    return arr

def maccs_bits_from_smiles(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return np.zeros(MACCS_BITS, dtype=np.uint8)
    fp = MACCSkeys.GenMACCSKeys(mol)
    # RDKit returns 167 bits; bit 0 is unused
    return np.array(fp, dtype=np.uint8)[1:]

map4_calc = MAP4(dimensions=MAP4_BITS, radius=MAP4_RADIUS) if MAP4_AVAILABLE else None

def map4_bits_from_smiles(smiles_str):
    if not MAP4_AVAILABLE:
        raise RuntimeError('MAP4 package is not available.')
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return np.zeros(MAP4_BITS, dtype=np.uint8)
    fp = np.asarray(map4_calc.calculate(mol), dtype=np.float32)
    if fp.shape[0] != MAP4_BITS:
        raise ValueError(f'MAP4 length mismatch: got {fp.shape[0]}, expected {MAP4_BITS}.')
    # Convert to binary bit activation for point-biserial bit-wise correlation
    return (fp != 0).astype(np.uint8)

X_ecfp4 = np.zeros((N, ECFP4_BITS), dtype=np.uint8)
X_maccs = np.zeros((N, MACCS_BITS), dtype=np.uint8)
X_map4 = np.zeros((N, MAP4_BITS), dtype=np.uint8) if MAP4_AVAILABLE else None

failed = {'ecfp4': 0, 'maccs': 0, 'map4': 0}

for i, s in enumerate(tqdm(smiles_list, desc='Computing aligned fingerprints')):
    ecfp4_row = ecfp4_bits_from_smiles(s)
    X_ecfp4[i] = ecfp4_row
    if ecfp4_row.sum() == 0 and s:
        failed['ecfp4'] += 1

    maccs_row = maccs_bits_from_smiles(s)
    X_maccs[i] = maccs_row
    if maccs_row.sum() == 0 and s:
        failed['maccs'] += 1

    if MAP4_AVAILABLE:
        map4_row = map4_bits_from_smiles(s)
        X_map4[i] = map4_row
        if map4_row.sum() == 0 and s:
            failed['map4'] += 1

print(f'X_desc shape:  {X_desc.shape}')
print(f'X_ecfp4 shape: {X_ecfp4.shape}')
print(f'X_maccs shape: {X_maccs.shape}')
if MAP4_AVAILABLE:
    print(f'X_map4 shape:  {X_map4.shape}')
print(f'Failed rows (all-zero fallback counts): {failed}')

Computing aligned fingerprints: 100%|██████████| 18242/18242 [01:26<00:00, 212.11it/s]

X_desc shape:  (18242, 201)
X_ecfp4 shape: (18242, 2048)
X_maccs shape: (18242, 166)
X_map4 shape:  (18242, 1024)
Failed rows (all-zero fallback counts): {'ecfp4': 0, 'maccs': 0, 'map4': 0}


In [5]:
# -----------------------------
# Point-biserial / Pearson-equivalent correlation
# -----------------------------
def compute_bit_descriptor_corr(X_bits, X_cont):
    B = X_bits.astype(np.float32)
    Y = X_cont.astype(np.float32)

    B_mean = B.mean(axis=0, keepdims=True)
    Y_mean = Y.mean(axis=0, keepdims=True)

    B_center = B - B_mean
    Y_center = Y - Y_mean

    B_std = B_center.std(axis=0, ddof=1)
    Y_std = Y_center.std(axis=0, ddof=1)

    cov = (B_center.T @ Y_center) / max(B.shape[0] - 1, 1)
    den = np.outer(B_std, Y_std)

    corr = np.full_like(cov, np.nan, dtype=np.float32)
    valid = den > 0
    corr[valid] = (cov[valid] / den[valid]).astype(np.float32)
    return corr

corr_ecfp4 = compute_bit_descriptor_corr(X_ecfp4, X_desc)
corr_maccs = compute_bit_descriptor_corr(X_maccs, X_desc)
corr_map4 = compute_bit_descriptor_corr(X_map4, X_desc) if MAP4_AVAILABLE else None

print(f'corr_ecfp4 shape: {corr_ecfp4.shape}')
print(f'corr_maccs shape: {corr_maccs.shape}')
if MAP4_AVAILABLE:
    print(f'corr_map4 shape:  {corr_map4.shape}')

corr_ecfp4 shape: (2048, 201)
corr_maccs shape: (166, 201)
corr_map4 shape:  (1024, 201)


In [6]:
# -----------------------------
# Best-matching descriptor per bit (argmax over abs correlation)
# -----------------------------
def best_descriptor_per_bit(corr):
    abs_corr = np.abs(corr)
    all_nan_rows = np.all(~np.isfinite(abs_corr), axis=1)

    best_idx = np.full((corr.shape[0],), -1, dtype=np.int32)
    valid_rows = ~all_nan_rows
    best_idx[valid_rows] = np.nanargmax(abs_corr[valid_rows], axis=1)

    best_corr = np.full((corr.shape[0],), np.nan, dtype=np.float32)
    best_abs_corr = np.full((corr.shape[0],), np.nan, dtype=np.float32)

    rows = np.arange(corr.shape[0])[valid_rows]
    cols = best_idx[valid_rows]
    best_corr[valid_rows] = corr[rows, cols]
    best_abs_corr[valid_rows] = abs_corr[rows, cols]

    return best_idx, best_corr, best_abs_corr

best_ecfp4_idx, best_ecfp4_corr, best_ecfp4_abs = best_descriptor_per_bit(corr_ecfp4)
best_maccs_idx, best_maccs_corr, best_maccs_abs = best_descriptor_per_bit(corr_maccs)
if MAP4_AVAILABLE:
    best_map4_idx, best_map4_corr, best_map4_abs = best_descriptor_per_bit(corr_map4)

print(f'ECFP4 valid best matches: {(best_ecfp4_idx >= 0).sum()} / {len(best_ecfp4_idx)}')
print(f'MACCS valid best matches: {(best_maccs_idx >= 0).sum()} / {len(best_maccs_idx)}')
if MAP4_AVAILABLE:
    print(f'MAP4 valid best matches:  {(best_map4_idx >= 0).sum()} / {len(best_map4_idx)}')

ECFP4 valid best matches: 2048 / 2048
MACCS valid best matches: 154 / 166
MAP4 valid best matches:  1024 / 1024


In [7]:
# -----------------------------
# Correctness checks
# -----------------------------
def count_constant_bits(X_bits):
    col_sum = X_bits.sum(axis=0)
    n = X_bits.shape[0]
    all_zero = int((col_sum == 0).sum())
    all_one = int((col_sum == n).sum())
    return all_zero, all_one

e0, e1 = count_constant_bits(X_ecfp4)
m0, m1 = count_constant_bits(X_maccs)
if MAP4_AVAILABLE:
    p0, p1 = count_constant_bits(X_map4)

print('Fingerprint/descriptor shapes:')
print(f'  ECFP4 corr expected ({ECFP4_BITS}, {D}), got {corr_ecfp4.shape}')
print(f'  MACCS corr expected ({MACCS_BITS}, {D}), got {corr_maccs.shape}')
if MAP4_AVAILABLE:
    print(f'  MAP4 corr expected ({MAP4_BITS}, {D}), got {corr_map4.shape}')
print('')
print('Constant bit counts:')
print(f'  ECFP4 all-zero={e0}, all-one={e1}')
print(f'  MACCS all-zero={m0}, all-one={m1}')
if MAP4_AVAILABLE:
    print(f'  MAP4  all-zero={p0}, all-one={p1}')

if D != 201:
    print(f'Warning: descriptor count is {D}, not 201. Check descriptor table overlap.')
if not MAP4_AVAILABLE:
    print('Warning: MAP4 package unavailable, MAP4 outputs were not generated.')

Fingerprint/descriptor shapes:
  ECFP4 corr expected (2048, 201), got (2048, 201)
  MACCS corr expected (166, 201), got (166, 201)
  MAP4 corr expected (1024, 201), got (1024, 201)

Constant bit counts:
  ECFP4 all-zero=0, all-one=0
  MACCS all-zero=12, all-one=0
  MAP4  all-zero=0, all-one=0


In [8]:
# -----------------------------
# Save outputs for downstream AUROC-join notebook
# -----------------------------
np.save(OUT_DIR / 'descriptor_matrix_train.npy', X_desc)
pd.Series(descriptor_cols, name='descriptor').to_csv(OUT_DIR / 'descriptor_columns_used.csv', index=False)

# Correlation matrices
np.save(OUT_DIR / 'corr_ecfp4_bits_x_descriptors.npy', corr_ecfp4)
np.save(OUT_DIR / 'corr_maccs_bits_x_descriptors.npy', corr_maccs)
if MAP4_AVAILABLE:
    np.save(OUT_DIR / 'corr_map4_bits_x_descriptors.npy', corr_map4)

# Best-match indices
np.save(OUT_DIR / 'best_descriptor_idx_ecfp4.npy', best_ecfp4_idx)
np.save(OUT_DIR / 'best_descriptor_idx_maccs.npy', best_maccs_idx)
if MAP4_AVAILABLE:
    np.save(OUT_DIR / 'best_descriptor_idx_map4.npy', best_map4_idx)

# Reusable bit-to-descriptor mapping tables
best_matches_ecfp4 = pd.DataFrame({
    'bit_index': np.arange(ECFP4_BITS, dtype=int),
    'best_descriptor_idx': best_ecfp4_idx,
    'best_descriptor': [descriptor_cols[i] if i >= 0 else None for i in best_ecfp4_idx],
    'best_corr': best_ecfp4_corr,
    'best_abs_corr': best_ecfp4_abs,
})
best_matches_ecfp4.to_csv(OUT_DIR / 'best_matches_ecfp4.csv', index=False)

best_matches_maccs = pd.DataFrame({
    'bit_index': np.arange(MACCS_BITS, dtype=int),
    'best_descriptor_idx': best_maccs_idx,
    'best_descriptor': [descriptor_cols[i] if i >= 0 else None for i in best_maccs_idx],
    'best_corr': best_maccs_corr,
    'best_abs_corr': best_maccs_abs,
})
best_matches_maccs.to_csv(OUT_DIR / 'best_matches_maccs.csv', index=False)

if MAP4_AVAILABLE:
    best_matches_map4 = pd.DataFrame({
        'bit_index': np.arange(MAP4_BITS, dtype=int),
        'best_descriptor_idx': best_map4_idx,
        'best_descriptor': [descriptor_cols[i] if i >= 0 else None for i in best_map4_idx],
        'best_corr': best_map4_corr,
        'best_abs_corr': best_map4_abs,
    })
    best_matches_map4.to_csv(OUT_DIR / 'best_matches_map4.csv', index=False)

summary_rows = [
    {'fingerprint': 'ecfp4', 'n_bits': ECFP4_BITS, 'n_descriptors': D, 'n_molecules': N},
    {'fingerprint': 'maccs', 'n_bits': MACCS_BITS, 'n_descriptors': D, 'n_molecules': N},
]
if MAP4_AVAILABLE:
    summary_rows.append({'fingerprint': 'map4', 'n_bits': MAP4_BITS, 'n_descriptors': D, 'n_molecules': N})
pd.DataFrame(summary_rows).to_csv(OUT_DIR / 'correlation_prep_summary.csv', index=False)

# Single compressed artifact for convenient downstream loading once AUROC values are ready
bundle_kwargs = {
    'descriptor_matrix_train': X_desc,
    'descriptor_columns': np.array(descriptor_cols, dtype=object),
    'corr_ecfp4': corr_ecfp4,
    'corr_maccs': corr_maccs,
    'best_descriptor_idx_ecfp4': best_ecfp4_idx,
    'best_descriptor_idx_maccs': best_maccs_idx,
}
if MAP4_AVAILABLE:
    bundle_kwargs['corr_map4'] = corr_map4
    bundle_kwargs['best_descriptor_idx_map4'] = best_map4_idx
np.savez_compressed(OUT_DIR / 'correlation_bundle_for_auroc_join.npz', **bundle_kwargs)

print('Saved output directory:')
print(OUT_DIR)

Saved output directory:
../results/cross_axis_correlation_prep


In [9]:
# Quick preview of top absolute matches per fingerprint
def preview_top_matches(best_idx, best_abs, fp_name, top_k=10):
    tmp = pd.DataFrame({
        'bit_index': np.arange(len(best_idx), dtype=int),
        'best_descriptor_idx': best_idx,
        'best_abs_corr': best_abs,
    })
    tmp = tmp[tmp['best_descriptor_idx'] >= 0].copy()
    tmp['best_descriptor'] = tmp['best_descriptor_idx'].map(lambda i: descriptor_cols[int(i)])
    tmp = tmp.sort_values('best_abs_corr', ascending=False).head(top_k).reset_index(drop=True)
    print(f'Top {top_k} matches for {fp_name}:')
    display(tmp)

preview_top_matches(best_ecfp4_idx, best_ecfp4_abs, 'ECFP4')
preview_top_matches(best_maccs_idx, best_maccs_abs, 'MACCS')
if MAP4_AVAILABLE:
    preview_top_matches(best_map4_idx, best_map4_abs, 'MAP4')

Top 10 matches for ECFP4:


,bit_index,best_descriptor_idx,best_abs_corr,best_descriptor
0,790,173,0.891879,fr_nitrile
1,1171,134,0.882895,fr_NH2
2,414,189,0.864057,fr_quatN
3,456,121,0.857421,fr_Ar_COO
4,561,104,0.846219,SlogP_VSA7
5,1143,198,0.814136,fr_unbrch_alkane
6,838,174,0.807098,fr_nitro
7,841,171,0.802389,fr_methoxy
8,1037,119,0.793465,fr_Al_OH_noTert
9,378,122,0.791689,fr_Ar_N


Top 10 matches for MACCS:


,bit_index,best_descriptor_idx,best_abs_corr,best_descriptor
0,40,173,0.969087,fr_nitrile
1,29,189,0.964419,fr_quatN
2,31,191,0.948337,fr_sulfonamd
3,55,174,0.943509,fr_nitro
4,15,155,0.932878,fr_epoxide
5,83,134,0.917971,fr_NH2
6,32,191,0.916748,fr_sulfonamd
7,62,174,0.912247,fr_nitro
8,124,62,0.859271,NumAromaticRings
9,28,184,0.848022,fr_phos_ester


Top 10 matches for MAP4:


,bit_index,best_descriptor_idx,best_abs_corr,best_descriptor
0,18,113,0.530766,VSA_EState6
1,655,14,0.518784,Chi1n
2,284,9,0.515999,BertzCT
3,267,9,0.512314,BertzCT
4,717,9,0.508805,BertzCT
5,771,14,0.508672,Chi1n
6,125,14,0.505072,Chi1n
7,555,53,0.504889,MolMR
8,142,14,0.501877,Chi1n
9,124,14,0.501551,Chi1n


In [10]:
# Optional manifest of generated files for downstream notebook
generated = sorted([p.name for p in OUT_DIR.glob('*')])
pd.DataFrame({'file': generated})

,file
0,best_descriptor_idx_ecfp4.npy
1,best_descriptor_idx_maccs.npy
2,best_descriptor_idx_map4.npy
3,best_matches_ecfp4.csv
4,best_matches_maccs.csv
5,best_matches_map4.csv
6,corr_ecfp4_bits_x_descriptors.npy
7,corr_maccs_bits_x_descriptors.npy
8,corr_map4_bits_x_descriptors.npy
9,correlation_bundle_for_auroc_join.npz


## Notes

- Correlation here is Pearson, which is mathematically equivalent to point-biserial when one variable is binary.
- Constant bits (all 0 or all 1) produce undefined correlations and are handled via NaN masking; those bits get best index = -1.
- This notebook only prepares correlation artifacts; join with per-bit AUROC belongs in a separate downstream notebook.